# Proyecto Final Big Data — Siniestralidad vial en Bogotá
## Cuaderno 2 · Almacenamiento en HDFS, MapReduce y análisis con Spark

**Entradas:** el CSV crudo (`historico_siniestros_bogota.csv`) y `interim_parquet.zip` (salida del cuaderno 1).

**Fases que cubre:** (1) capa cruda en HDFS, (2) procesamiento con Hadoop MapReduce (Streaming, con combiner), (3) análisis exploratorio con Spark, (4) modelo para predecir severidad y (5) zonas calientes con K-Means.

**Salida:** `resultados_cuaderno2.zip` (tablas y gráficos para la presentación).

Alcance: HDFS corre como servicio real (NameNode + DataNode, un solo nodo). Los trabajos MapReduce usan el ejecutor local de Hadoop, sin YARN: es una sola máquina (Colab), no un clúster.

## Entradas

In [ ]:
import os, shutil, zipfile
from pathlib import Path

BASE = Path('/content/proyecto_bigdata')
BASE.mkdir(parents=True, exist_ok=True)
CSV_LOCAL = '/content/historico_siniestros_bogota.csv'

# Java por defecto de Colab (lo usa Spark); se guarda antes de instalar Java 11 para Hadoop
JAVA_SPARK_HOME = str(Path(os.path.realpath(shutil.which('java'))).parent.parent)
print('Java de Spark:', JAVA_SPARK_HOME)

from google.colab import files   # solo se usa para descargar resultados al final

if not Path(CSV_LOCAL).exists():
    raise FileNotFoundError(
        'Falta historico_siniestros_bogota.csv. Arrástralo al panel de Archivos (izquierda, carpeta /content) '
        'y vuelve a ejecutar esta celda.')

if not (BASE / 'interim' / 'siniestros_limpio').exists():
    zips = [p for p in (Path('/content/interim_parquet.zip'), Path('interim_parquet.zip')) if p.exists()]
    if not zips:
        raise FileNotFoundError(
            'Falta interim_parquet.zip (se descarga al final del cuaderno 1). Arrástralo al panel de Archivos '
            '(izquierda, carpeta /content) y vuelve a ejecutar esta celda. Evita files.upload(): es muy lento.')
    zipfile.ZipFile(zips[0]).extractall(BASE)
    print('Parquet extraído desde', zips[0])

print('CSV:', Path(CSV_LOCAL).exists(), '| Parquet limpio:', (BASE / 'interim' / 'siniestros_limpio').exists())

---
# Almacenamiento y procesamiento: HDFS + Hadoop MapReduce

## 0. Utilidades y rutas

In [ ]:
import os, subprocess, time, textwrap, shutil
from pathlib import Path

BASE = Path('/content/proyecto_bigdata')
MR = BASE / 'mr'
OUT = BASE / 'out'
for d in (MR, OUT):
    d.mkdir(parents=True, exist_ok=True)

CSV_LOCAL = '/content/historico_siniestros_bogota.csv'
HADOOP_HOME = '/opt/hadoop'
HDFS_RAW = '/user/colab/siniestros/raw'
HDFS_OUT = '/user/colab/siniestros/mr_out'

def sh(cmd, env=None, check=True, tail=25, show=True):
    e = {**os.environ, **(env or {})}
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=e)
    salida = (r.stdout + r.stderr).strip().splitlines()
    if show and salida:
        print('\n'.join(salida[-tail:]))
    if check and r.returncode != 0:
        raise RuntimeError(f'Falló (código {r.returncode}): {cmd}')
    return r

## 1. Java 11 y Hadoop 3.3.6
Hadoop 3.3.6 es la combinación soportada con Java 11; se instala solo para Hadoop (Spark sigue con el Java de Colab).
La descarga pesa ~700 MB y puede tardar varios minutos.

In [ ]:
sh('apt-get -qq update && apt-get -qq install -y openjdk-11-jdk-headless', tail=3)
JAVA11 = '/usr/lib/jvm/java-11-openjdk-amd64'
assert Path(JAVA11).exists(), 'No se instaló Java 11'

if not Path(f'{HADOOP_HOME}/bin/hdfs').exists():
    urls = [
        'https://dlcdn.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz',
        'https://downloads.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz',
        'https://archive.apache.org/dist/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz',
    ]
    for u in urls:
        print('Intentando', u)
        if sh(f'curl -fL --retry 3 -o /tmp/hadoop.tgz {u}', check=False, show=False).returncode == 0:
            break
    else:
        raise RuntimeError('No se pudo descargar Hadoop desde ningún espejo')
    sh('tar -xzf /tmp/hadoop.tgz -C /opt && mv /opt/hadoop-3.3.6 /opt/hadoop && rm /tmp/hadoop.tgz', show=False)

print(sh(f'JAVA_HOME={JAVA11} {HADOOP_HOME}/bin/hadoop version', show=False).stdout.splitlines()[0])

## 2. Configuración pseudo-distribuida (un nodo)

In [ ]:
conf = Path(HADOOP_HOME) / 'etc/hadoop'

(conf / 'hadoop-env.sh').write_text(textwrap.dedent(f'''
    export JAVA_HOME={JAVA11}
    export HDFS_NAMENODE_USER=root
    export HDFS_DATANODE_USER=root
    export HDFS_SECONDARYNAMENODE_USER=root
'''))

def xml(props):
    cuerpo = ''.join(f'<property><name>{k}</name><value>{v}</value></property>' for k, v in props.items())
    return f'<?xml version="1.0"?><configuration>{cuerpo}</configuration>'

(conf / 'core-site.xml').write_text(xml({'fs.defaultFS': 'hdfs://localhost:9000'}))
(conf / 'hdfs-site.xml').write_text(xml({
    'dfs.replication': '1',
    'dfs.namenode.name.dir': 'file:///content/hdfs/nn',
    'dfs.datanode.data.dir': 'file:///content/hdfs/dn',
}))
(conf / 'mapred-site.xml').write_text(xml({'mapreduce.framework.name': 'local'}))

HENV = {'JAVA_HOME': JAVA11, 'HADOOP_HOME': HADOOP_HOME,
        'PATH': f'{HADOOP_HOME}/bin:{HADOOP_HOME}/sbin:' + os.environ['PATH']}
print('Configuración escrita en', conf)

## 3. Arranque de HDFS (NameNode + DataNode)

In [ ]:
shutil.rmtree('/content/hdfs', ignore_errors=True)
sh('hdfs namenode -format -force -nonInteractive', env=HENV, tail=3)
sh('hdfs --daemon start namenode', env=HENV, check=False)
sh('hdfs --daemon start datanode', env=HENV, check=False)

for _ in range(30):                       # espera a que el DataNode se registre
    r = sh('hdfs dfsadmin -report', env=HENV, check=False, show=False)
    if 'Live datanodes (1)' in r.stdout:
        break
    time.sleep(2)
else:
    raise RuntimeError('HDFS no levantó: revisa /opt/hadoop/logs')

print('\n'.join(l for l in r.stdout.splitlines()
                if any(k in l for k in ('Configured Capacity', 'DFS Used', 'Live datanodes', 'Name:'))))

## 4. Carga de la capa cruda a HDFS

In [ ]:
sh(f'hdfs dfs -mkdir -p {HDFS_RAW}', env=HENV, show=False)
sh(f'hdfs dfs -put -f {CSV_LOCAL} {HDFS_RAW}/', env=HENV, show=False)
sh(f'hdfs dfs -ls -h {HDFS_RAW}', env=HENV)
print()
sh(f'hdfs fsck {HDFS_RAW} -files -blocks 2>&1 | grep -E "Total|Under-replicated|Status|BP-"', env=HENV, tail=10)

## 5. Mapper y reducer (Hadoop Streaming)
Un mapper genérico recibe las columnas que forman la clave (`KEYS`); el reducer suma conteos y se reutiliza como combiner (la suma es asociativa).

In [ ]:
MAPPER_SRC = '#!/usr/bin/env python3\n"""Mapper genérico de Hadoop Streaming para el Histórico de Siniestros.\nEmite \'clave1|clave2<TAB>1\' según las columnas pedidas en la variable KEYS."""\nimport csv, os, sys\n\nSEP = os.environ.get(\'SEP\', \';\')\nKEYS = [k.strip() for k in os.environ.get(\'KEYS\', \'LOCALIDAD,GRAVEDAD\').split(\',\')]\n\n# Posiciones de columnas del CSV (orden verificado en el perfilado)\nCOLS = {\'X\': 0, \'Y\': 1, \'OBJECTID\': 2, \'FORMULARIO\': 3, \'CODIGO_ACCIDENTE\': 4,\n        \'FECHA_OCURRENCIA_ACC\': 5, \'ANO_OCURRENCIA_ACC\': 6, \'DIRECCION\': 7,\n        \'GRAVEDAD\': 8, \'CLASE_ACC\': 9, \'LOCALIDAD\': 10, \'FECHA_HORA_ACC\': 11}\n\ndef valor(fila, nombre):\n    if nombre == \'HORA\':                        # derivada de FECHA_HORA_ACC (yyyy/MM/dd HH:mm:ss+00)\n        return fila[COLS[\'FECHA_HORA_ACC\']][11:13]\n    return fila[COLS[nombre]].strip().upper()\n\nif hasattr(sys.stdin, \'reconfigure\'):\n    sys.stdin.reconfigure(encoding=\'utf-8\', errors=\'replace\')\n\nfor fila in csv.reader(sys.stdin, delimiter=SEP):\n    if len(fila) < 12 or fila[COLS[\'CODIGO_ACCIDENTE\']].lstrip(\'\\ufeff\') == \'CODIGO_ACCIDENTE\':\n        continue                                 # encabezado o línea malformada\n    clave = \'|\'.join(valor(fila, k) for k in KEYS)\n    print(f\'{clave}\\t1\')\n'
REDUCER_SRC = '#!/usr/bin/env python3\n"""Reducer (y combiner): suma los conteos por clave."""\nimport sys\n\nactual, total = None, 0\nfor linea in sys.stdin:\n    clave, _, n = linea.rstrip(\'\\n\').rpartition(\'\\t\')\n    if clave != actual:\n        if actual is not None:\n            print(f\'{actual}\\t{total}\')\n        actual, total = clave, 0\n    total += int(n)\nif actual is not None:\n    print(f\'{actual}\\t{total}\')\n'
(MR / 'mapper.py').write_text(MAPPER_SRC)
(MR / 'reducer.py').write_text(REDUCER_SRC)
for f in ('mapper.py', 'reducer.py'):
    (MR / f).chmod(0o755)
print('Escritos:', [p.name for p in MR.iterdir()])

## 6. Trabajos MapReduce

In [ ]:
import glob, re
import pandas as pd

jar = glob.glob(f'{HADOOP_HOME}/share/hadoop/tools/lib/hadoop-streaming-*.jar')[0]
SEP = ';' if open(CSV_LOCAL, 'rb').readline().decode('utf-8', 'ignore').count(';') > 5 else ','

CONTADORES = ('Map input records', 'Map output records', 'Combine input records',
              'Combine output records', 'Reduce input groups', 'Reduce output records')

def run_job(nombre, claves):
    destino = f'{HDFS_OUT}/{nombre}'
    sh(f'hdfs dfs -rm -r -f {destino}', env=HENV, show=False)
    cmd = (f'hadoop jar {jar} '
           f'-D mapreduce.job.name={nombre} '
           f'-cmdenv SEP="{SEP}" -cmdenv KEYS={claves} '
           f'-input {HDFS_RAW} -output {destino} '
           f'-mapper "python3 {MR}/mapper.py" '
           f'-combiner "python3 {MR}/reducer.py" '
           f'-reducer "python3 {MR}/reducer.py"')
    t0 = time.perf_counter()
    r = sh(cmd, env=HENV, show=False)
    seg = time.perf_counter() - t0
    texto = r.stdout + r.stderr
    cont = {}
    for linea in texto.splitlines():
        for c in CONTADORES:
            if c in linea:
                cont[c] = int(re.search(r'=(\d+)', linea).group(1))
    crudo = sh(f'hdfs dfs -cat {destino}/part-*', env=HENV, show=False).stdout
    filas = [l.rsplit('\t', 1) for l in crudo.strip().splitlines()]
    df = pd.DataFrame(filas, columns=['clave', 'n'])
    df['n'] = df['n'].astype(int)
    df[claves.split(',')] = df['clave'].str.split('|', expand=True)
    return df.drop(columns='clave'), seg, cont

resultados, tiempos, contadores = {}, {}, {}
for nombre, claves in {'localidad_gravedad': 'LOCALIDAD,GRAVEDAD',
                       'anio_clase': 'ANO_OCURRENCIA_ACC,CLASE_ACC',
                       'hora_gravedad': 'HORA,GRAVEDAD'}.items():
    resultados[nombre], tiempos[nombre], contadores[nombre] = run_job(nombre, claves)
    print(f'{nombre:20s} {tiempos[nombre]:6.1f} s | {contadores[nombre]}')

## 7. Validación y resultados

In [ ]:
n_lineas = sum(1 for _ in open(CSV_LOCAL, 'rb')) - 1     # sin encabezado
print('Registros en el CSV (sin encabezado):', n_lineas)
for nombre, df in resultados.items():
    print(f'{nombre:20s} suma de conteos = {df.n.sum()}  ->', 'OK' if df.n.sum() == n_lineas else 'REVISAR')

res = pd.DataFrame({'segundos': tiempos}).join(pd.DataFrame(contadores).T)
res.to_csv(OUT / 'mapreduce_metricas.csv')
res

In [ ]:
import matplotlib.pyplot as plt

lg = resultados['localidad_gravedad'].pivot_table(index='LOCALIDAD', columns='GRAVEDAD', values='n', fill_value=0)
lg['total'] = lg.sum(axis=1)
lg = lg.sort_values('total', ascending=True).drop(columns='total')
ax = lg.plot(kind='barh', stacked=True, figsize=(9, 7))
ax.set_title('Siniestros por localidad y gravedad (MapReduce)')
ax.set_xlabel('Siniestros'); ax.set_ylabel('')
plt.tight_layout(); plt.savefig(OUT / 'mr_localidad_gravedad.png', dpi=150); plt.show()

hg = resultados['hora_gravedad'].pivot_table(index='HORA', columns='GRAVEDAD', values='n', fill_value=0).sort_index()
ax = hg.plot(kind='line', marker='o', figsize=(9, 4))
ax.set_title('Siniestros por hora del día y gravedad (MapReduce)')
ax.set_xlabel('Hora'); ax.set_ylabel('Siniestros')
plt.tight_layout(); plt.savefig(OUT / 'mr_hora_gravedad.png', dpi=150); plt.show()

ac = resultados['anio_clase'].pivot_table(index='ANO_OCURRENCIA_ACC', columns='CLASE_ACC', values='n', fill_value=0).sort_index()
ax = ac.plot(kind='bar', stacked=True, figsize=(9, 4))
ax.set_title('Siniestros por año y clase (MapReduce)')
ax.set_xlabel('Año'); ax.set_ylabel('Siniestros')
plt.tight_layout(); plt.savefig(OUT / 'mr_anio_clase.png', dpi=150); plt.show()

for nombre, df in resultados.items():
    df.to_csv(OUT / f'mr_{nombre}.csv', index=False)
print('Guardado en', OUT)

---
# Análisis y modelado con Apache Spark

## 0. Spark y datos

In [ ]:
import os
os.environ['JAVA_HOME'] = JAVA_SPARK_HOME   # Spark usa el Java por defecto, no el 11 de Hadoop

from pyspark.sql import SparkSession, functions as F, Window
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

spark = (SparkSession.builder.master('local[*]').appName('analisis_siniestros')
         .config('spark.sql.session.timeZone', 'America/Bogota')
         .config('spark.sql.shuffle.partitions', '8')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')

BASE = Path('/content/proyecto_bigdata')
OUT = BASE / 'out'
OUT.mkdir(parents=True, exist_ok=True)

df = (spark.read.parquet(f'file://{BASE}/interim/siniestros_limpio')
      .withColumn('dow', F.dayofweek('fecha'))
      .withColumn('grave', F.col('es_grave').cast('int'))
      .fillna({'localidad': 'SIN DATO', 'clase_acc': 'OTRO'})
      .cache())
n = df.count()
print('registros:', n)
df.agg(F.min('fecha').alias('desde'), F.max('fecha').alias('hasta')).show()

## 1. Cobertura temporal y tendencia anual
Se revisa el último mes con datos: si 2021 es parcial, no se compara año contra año.

In [ ]:
mensual = (df.groupBy('anio', 'mes').count().orderBy('anio', 'mes').toPandas())
ult = mensual.tail(3)
print('Últimos meses con datos:'); print(ult.to_string(index=False))

anual = (df.groupBy('anio')
         .agg(F.count('*').alias('siniestros'),
              F.round(F.avg('grave') * 100, 1).alias('pct_graves'),
              F.sum(F.when(F.col('gravedad') == 'CON MUERTOS', 1).otherwise(0)).alias('con_muertos'))
         .orderBy('anio').toPandas())
anual['var_anual_%'] = (anual['siniestros'].pct_change() * 100).round(1)
anual

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(anual['anio'].astype(str), anual['siniestros'], color='#4C78A8')
ax.set_ylabel('Siniestros'); ax.set_title('Siniestros por año y % de graves')
ax2 = ax.twinx()
ax2.plot(anual['anio'].astype(str), anual['pct_graves'], color='#E45756', marker='o')
ax2.set_ylabel('% graves'); ax2.set_ylim(0, max(60, anual['pct_graves'].max() + 5))
plt.tight_layout(); plt.savefig(OUT / 'an_tendencia_anual.png', dpi=150); plt.show()
anual.to_csv(OUT / 'an_anual.csv', index=False)

## 2. Localidades críticas
Top por volumen y por proporción de graves (solo localidades con al menos 500 siniestros, para evitar tasas inestables).

In [ ]:
loc = (df.groupBy('localidad')
       .agg(F.count('*').alias('siniestros'),
            F.round(F.avg('grave') * 100, 1).alias('pct_graves'),
            F.sum(F.when(F.col('gravedad') == 'CON MUERTOS', 1).otherwise(0)).alias('con_muertos'))
       .toPandas())
min_n = 500 if n > 5000 else 1
loc = loc[loc.siniestros >= min_n]
top_vol = loc.sort_values('siniestros', ascending=False).head(10)
top_gra = loc.sort_values('pct_graves', ascending=False).head(10)
display(top_vol); display(top_gra)

fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
top_vol.sort_values('siniestros').plot.barh(x='localidad', y='siniestros', ax=axs[0], legend=False, color='#4C78A8')
axs[0].set_title('Top 10 por volumen'); axs[0].set_ylabel('')
top_gra.sort_values('pct_graves').plot.barh(x='localidad', y='pct_graves', ax=axs[1], legend=False, color='#E45756')
axs[1].set_title('Top 10 por % de graves'); axs[1].set_ylabel('')
plt.tight_layout(); plt.savefig(OUT / 'an_localidades.png', dpi=150); plt.show()
loc.to_csv(OUT / 'an_localidades.csv', index=False)

## 3. Franjas del día y mapa de calor hora × día de la semana

In [ ]:
franja = (df.groupBy('franja')
          .agg(F.count('*').alias('siniestros'),
               F.round(F.avg('grave') * 100, 1).alias('pct_graves'),
               F.round(F.avg(F.when(F.col('gravedad') == 'CON MUERTOS', 1).otherwise(0)) * 100, 2).alias('pct_muertos'))
          .toPandas().set_index('franja').reindex(['MADRUGADA', 'MANANA', 'TARDE', 'NOCHE']))
display(franja)

nombres = {2: 'Lun', 3: 'Mar', 4: 'Mie', 5: 'Jue', 6: 'Vie', 7: 'Sab', 1: 'Dom'}
hm = (df.groupBy('dow', 'hora').count().toPandas()
      .pivot(index='dow', columns='hora', values='count').fillna(0)
      .reindex([2, 3, 4, 5, 6, 7, 1]))
hm.index = [nombres[i] for i in hm.index]
fig, ax = plt.subplots(figsize=(11, 3.8))
im = ax.imshow(hm.values, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(hm.shape[1])); ax.set_xticklabels(hm.columns)
ax.set_yticks(range(hm.shape[0])); ax.set_yticklabels(hm.index)
ax.set_xlabel('Hora del día'); ax.set_title('Siniestros por día de la semana y hora')
fig.colorbar(im, ax=ax, label='Siniestros')
plt.tight_layout(); plt.savefig(OUT / 'an_heatmap_hora_dia.png', dpi=150); plt.show()
franja.to_csv(OUT / 'an_franja.csv')

## 4. Modelo: ¿el siniestro será grave?
Variable objetivo: `grave` = con heridos o con muertos (≈36 % de los casos).
Variables: hora (codificada con seno/coseno), mes, fin de semana, franja, localidad y clase de accidente (las clases con muy pocos casos se agrupan en `OTRO`).
Validación **temporal**: entrenamiento con los años anteriores a 2021 y prueba con 2021, para simular predecir hacia adelante.

**Alcance honesto del modelo:** la clase de accidente (p. ej. atropello) se conoce una vez ocurrido el siniestro, así que el modelo sirve para *clasificar la severidad esperada y entender factores de riesgo* cuando se reporta un evento, no para anticipar un siniestro antes de que ocurra.

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
import math

raras = [r['clase_acc'] for r in df.groupBy('clase_acc').count().filter('count < 100').collect()]
datos = (df
    .withColumn('clase_agr', F.when(F.col('clase_acc').isin(raras), 'OTRO').otherwise(F.col('clase_acc')))
    .withColumn('hora_sin', F.sin(2 * math.pi * F.col('hora') / 24))
    .withColumn('hora_cos', F.cos(2 * math.pi * F.col('hora') / 24))
    .withColumn('fds', F.col('fin_de_semana').cast('int'))
    .select('grave', 'anio', 'localidad', 'clase_agr', 'franja', 'mes', 'fds', 'hora_sin', 'hora_cos'))

ultimo = datos.agg(F.max('anio')).first()[0]
train = datos.filter(F.col('anio') < ultimo)
test = datos.filter(F.col('anio') == ultimo)
print(f'train: {train.count()} (hasta {ultimo - 1}) | test: {test.count()} (año {ultimo})')

pos = train.filter('grave = 1').count() / train.count()
print(f'proporción de graves en train: {pos:.3f}')
train = train.withColumn('peso', F.when(F.col('grave') == 1, 0.5 / pos).otherwise(0.5 / (1 - pos)))

cat = ['localidad', 'clase_agr', 'franja']
indexers = [StringIndexer(inputCol=c, outputCol=c + '_i', handleInvalid='keep') for c in cat]
encoder = OneHotEncoder(inputCols=[c + '_i' for c in cat], outputCols=[c + '_v' for c in cat], handleInvalid='keep')
asm = VectorAssembler(inputCols=[c + '_v' for c in cat] + ['mes', 'fds', 'hora_sin', 'hora_cos'], outputCol='features')

lr = LogisticRegression(labelCol='grave', weightCol='peso', maxIter=50, regParam=0.01)
rf = RandomForestClassifier(labelCol='grave', numTrees=60, maxDepth=8, seed=7, weightCol='peso')
modelos = {'Regresión logística': Pipeline(stages=indexers + [encoder, asm, lr]).fit(train),
           'Random Forest': Pipeline(stages=indexers + [encoder, asm, rf]).fit(train)}

In [ ]:
ev_roc = BinaryClassificationEvaluator(labelCol='grave', metricName='areaUnderROC')
ev_pr = BinaryClassificationEvaluator(labelCol='grave', metricName='areaUnderPR')

filas, preds = [], {}
base_pr = test.filter('grave = 1').count() / test.count()
for nombre, m in modelos.items():
    p = m.transform(test).cache()
    preds[nombre] = p
    tp = p.filter('grave = 1 and prediction = 1').count()
    fp = p.filter('grave = 0 and prediction = 1').count()
    fn = p.filter('grave = 1 and prediction = 0').count()
    tn = p.filter('grave = 0 and prediction = 0').count()
    filas.append({'modelo': nombre,
                  'AUC_ROC': round(ev_roc.evaluate(p), 3),
                  'AUC_PR': round(ev_pr.evaluate(p), 3),
                  'precision': round(tp / max(tp + fp, 1), 3),
                  'recall': round(tp / max(tp + fn, 1), 3),
                  'exactitud': round((tp + tn) / max(tp + tn + fp + fn, 1), 3),
                  'TP': tp, 'FP': fp, 'FN': fn, 'TN': tn})
metricas = pd.DataFrame(filas).set_index('modelo')
print(f'Línea base: AUC_ROC = 0.5 | AUC_PR = proporción de graves en test = {base_pr:.3f}')
metricas

In [ ]:
# Importancia de variables del Random Forest
rf_model = modelos['Random Forest']
cols = rf_model.transform(test.limit(1)).schema['features'].metadata['ml_attr']['attrs']
attrs = sorted([a for grupo in cols.values() for a in grupo], key=lambda a: a['idx'])
imp = pd.Series(rf_model.stages[-1].featureImportances.toArray(), index=[a['name'] for a in attrs])
imp_top = imp.sort_values(ascending=False).head(12)
display(imp_top.round(4).to_frame('importancia'))

ax = imp_top.sort_values().plot.barh(figsize=(8, 4.5), color='#54A24B')
ax.set_title('Importancia de variables (Random Forest)')
plt.tight_layout(); plt.savefig(OUT / 'an_importancia_variables.png', dpi=150); plt.show()
metricas.to_csv(OUT / 'an_modelo_metricas.csv')

## 5. Zonas calientes con K-Means (coordenadas)
Se prueban varios `k` y se elige por silueta; luego se perfila cada zona (volumen, % graves, localidad predominante).

In [ ]:
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

geo = df.filter('coord_valida = true').select('latitud', 'longitud', 'grave', 'gravedad', 'localidad')
geo_v = VectorAssembler(inputCols=['latitud', 'longitud'], outputCol='features').transform(geo).cache()
n_geo = geo_v.count()
print('registros con coordenadas válidas:', n_geo)

ks = [k for k in (6, 8, 10, 12) if k < n_geo]
ev = ClusteringEvaluator(featuresCol='features', metricName='silhouette', distanceMeasure='squaredEuclidean')
sil = {}
for k in ks:
    sil[k] = ev.evaluate(KMeans(k=k, seed=7, featuresCol='features').fit(geo_v).transform(geo_v))
    print(f'k={k:2d}  silueta={sil[k]:.3f}')
k_best = max(sil, key=sil.get)
print('k elegido:', k_best)

In [ ]:
km = KMeans(k=k_best, seed=7, featuresCol='features').fit(geo_v)
cl = km.transform(geo_v).withColumnRenamed('prediction', 'zona')

perfil = (cl.groupBy('zona')
          .agg(F.count('*').alias('siniestros'),
               F.round(F.avg('grave') * 100, 1).alias('pct_graves'),
               F.sum(F.when(F.col('gravedad') == 'CON MUERTOS', 1).otherwise(0)).alias('con_muertos'),
               F.round(F.avg('latitud'), 4).alias('lat_centro'),
               F.round(F.avg('longitud'), 4).alias('lon_centro'))
          .toPandas())
dom = (cl.groupBy('zona', 'localidad').count()
       .withColumn('r', F.row_number().over(Window.partitionBy('zona').orderBy(F.desc('count'))))
       .filter('r = 1').select('zona', F.col('localidad').alias('localidad_predominante')).toPandas())
perfil = perfil.merge(dom, on='zona').sort_values('siniestros', ascending=False)
display(perfil)

muestra = cl.select('latitud', 'longitud', 'zona', 'grave').toPandas()
if len(muestra) > 40000:
    muestra = muestra.sample(40000, random_state=7)
fig, ax = plt.subplots(figsize=(7, 7))
sc = ax.scatter(muestra.longitud, muestra.latitud, c=muestra.zona, s=2, cmap='tab10', alpha=0.5)
ax.scatter(perfil.lon_centro, perfil.lat_centro, c='black', marker='x', s=80)
ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud'); ax.set_title(f'Zonas calientes (K-Means, k={k_best})')
plt.tight_layout(); plt.savefig(OUT / 'an_zonas_kmeans.png', dpi=150); plt.show()
perfil.to_csv(OUT / 'an_zonas_perfil.csv', index=False)

## 6. Empaquetar resultados

In [ ]:
import zipfile, json
resumen = {'registros': n, 'k_kmeans': int(k_best), 'silueta': round(sil[k_best], 3),
           'modelos': metricas.reset_index().to_dict(orient='records')}
(OUT / 'an_resumen.json').write_text(json.dumps(resumen, ensure_ascii=False, indent=2))

zpath = OUT / 'resultados_cuaderno2.zip'
with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as z:
    for patron in ('an_*', 'mr_*', 'mapreduce_metricas.csv'):
        for f in sorted(OUT.glob(patron)):
            z.write(f, f.name)
print(zpath, f'{zpath.stat().st_size/1e6:.2f} MB')
files.download(str(zpath))